# CER — Cluster Experiment Runner Demo

This notebook demonstrates the full CER workflow:

1. **Build** the Singularity container locally
2. **Create a workspace** for the agent
3. **Edit** experiment config (Hydra)
4. **Submit** to the cluster via MCP
5. **Monitor** status and query W&B results
6. **Reset** the workspace when done

Architecture:
- The experiment code and agent run **inside** a Singularity/Apptainer container
- `cer-mcp` runs on the **host** with SSH access to the cluster
- The agent connects to the MCP server over `localhost` (Apptainer shares host networking)
- Each agent gets its own **workspace** (git worktree + branch) for parallel work

## 0. Build the Container

The container is defined in `experiment.def` and provides the full runtime environment
(PyTorch, W&B, your pip dependencies). The same container runs locally and on the cluster.

**Prerequisites:** [Apptainer](https://apptainer.org/docs/admin/main/installation.html) installed on your machine.

In [ ]:
# Check Apptainer is installed
!apptainer --version

In [ ]:
# Build the container from the definition file
# This pulls the base image and installs dependencies (~10-15 min first time, ~9GB)
!apptainer build experiment.sif experiment.def

In [ ]:
# Verify the container works
!apptainer exec experiment.sif python -c "import torch; print(f'PyTorch {torch.__version__}')"

### Running inside the container

```bash
# Interactive shell (for the agent or manual work)
apptainer exec --nv --bind .:/workspace --pwd /workspace experiment.sif bash

# With GPU (--nv) and workspace mounted:
#   - Your code is at /workspace
#   - Host networking is shared (agent can reach cer-mcp on localhost)
#   - No SSH access inside the container
```

**Rebuild** only when `experiment.def` changes (new system/pip deps). Code changes don't require a rebuild — the workspace is bind-mounted.

## 1. Start the MCP Server

In a separate terminal on the **host** (outside the container), run:
```bash
cer-mcp
```

This starts the server on `http://localhost:8000/sse`.

Inside the container, the `./cer` CLI client connects to it over localhost.

In [ ]:
import subprocess

CER = "./cer"

# Check it works
!{CER} --help

## 2. Create a Workspace

Each agent gets its own workspace — a git worktree on a separate branch.
Multiple agents can work in parallel, each in their own workspace.

The max number of workspaces is controlled by `local.max_workspaces` in `cer.yaml`.

In [ ]:
# Create a workspace for this agent
!{CER} workspace create agent-001

In [ ]:
# List all active workspaces
!{CER} workspace list

In [ ]:
# The agent works inside the workspace directory
import os
WORKSPACE = "workspaces/agent-001"
os.chdir(WORKSPACE)
print(f"Working in: {os.getcwd()}")

## 3. Edit Experiment Config

The experiment uses [Hydra](https://hydra.cc/) for configuration.
The agent modifies `configs/config.yaml` to change hyperparameters.

The full Hydra config is automatically logged to W&B.
Files listed in `save_artifacts` are saved as W&B artifacts for reproducibility
(so they survive workspace cleanup).

In [ ]:
# View current config
!cat configs/config.yaml

In [ ]:
# Example: agent modifies hyperparameters
import yaml

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["model"]["hidden_dim"] = 256
cfg["model"]["lr"] = 5e-4
cfg["training"]["max_epochs"] = 20

with open("configs/config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Updated config:")
!cat configs/config.yaml

## 4. Commit, Push & Submit

CER uses **commit = experiment**. Every experiment is identified by a git commit hash.

In [ ]:
# Commit and push changes on the agent's branch
!git add -A && git commit -m "experiment: hidden_dim=256, lr=5e-4, epochs=20"
!git push origin agent-001

In [ ]:
# Get the commit hash
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"Commit: {COMMIT}")

In [ ]:
# Submit to the cluster
!{CER} submit {COMMIT}

In [ ]:
JOB_ID = input("Enter job ID from above: ")

## 5. Check Status

Queries SLURM via SSH. Run multiple times to watch: `PENDING` → `RUNNING` → `COMPLETED`.

In [ ]:
!{CER} status {JOB_ID}

## 6. List All Experiments

In [ ]:
!{CER} list

## 7. Query W&B Results

Finds the W&B run by commit hash tag, pulls summary and history.
The Hydra config and `save_artifacts` files are stored as W&B artifacts.

In [ ]:
# Summary metrics
!{CER} results {JOB_ID}

In [ ]:
# Full training history
!{CER} results {JOB_ID} --history

In [ ]:
# Filtered metrics
!{CER} results {JOB_ID} --history --key train/loss --key val/loss

## 8. Programmatic Access (JSON)

Parse the JSON output for plotting or automated pipelines.

In [ ]:
import json

raw = subprocess.check_output([CER, "results", JOB_ID, "--history"]).decode()
data = json.loads(raw)

print(f"Run:     {data['wandb']['run_name']} ({data['wandb']['state']})")
print(f"URL:     {data['wandb']['url']}")
print(f"Summary: {data['wandb']['summary']}")
print(f"History: {len(data.get('history', []))} steps")

## 9. Plot Results

In [ ]:
import matplotlib.pyplot as plt

history = data["history"]
steps = range(len(history))
train_loss = [h.get("train/loss") for h in history]
val_loss = [h.get("val/loss") for h in history]

plt.figure(figsize=(10, 5))
if any(v is not None for v in train_loss):
    plt.plot(steps, train_loss, label="train/loss", marker="o")
if any(v is not None for v in val_loss):
    plt.plot(steps, val_loss, label="val/loss", marker="s")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title(f"Experiment {JOB_ID} (commit {COMMIT[:8]})")
plt.legend()
plt.grid(True)
plt.show()

## 10. Reset Workspace

When the agent is done with an experiment line, reset the workspace to start fresh from main.

This deletes:
- The local worktree and branch
- The remote branch

Then recreates a clean worktree from latest main.

**Important:** The Hydra config and files listed in `save_artifacts` are already saved as
W&B artifacts by the training script, so nothing is lost.

In [ ]:
# Go back to repo root before resetting
os.chdir("../..")

# Reset the workspace
!{CER} workspace reset agent-001

In [ ]:
# Verify it was recreated fresh
!{CER} workspace list

## 11. Cancel an Experiment

In [ ]:
# Uncomment to cancel a running job:
# !{CER} cancel {JOB_ID}

---

## Command Reference

The `./cer` CLI client connects to the MCP server (`cer-mcp`) running on the host.

| Command | Description |
|---------|-------------|
| `./cer workspace create <name>` | Create a new workspace (worktree + branch) |
| `./cer workspace list` | List all active workspaces |
| `./cer workspace reset <name>` | Delete and recreate workspace from latest main |
| `./cer submit <commit>` | Submit experiment to SLURM cluster |
| `./cer status <job_id>` | Check SLURM job status (JSON) |
| `./cer cancel <job_id>` | Cancel a running job |
| `./cer results <job_id>` | W&B summary metrics (JSON) |
| `./cer results <job_id> --history` | Full metric history |
| `./cer results <job_id> --key <metric>` | Filter specific metrics |
| `./cer list` | List all experiments |

### Hydra Config

Edit `configs/config.yaml` to change experiment hyperparameters. The full config
is logged to W&B automatically. Files listed in `save_artifacts` are saved as
W&B artifacts for reproducibility after workspace cleanup.